In [ ]:
import base64
import os
import textwrap
from pathlib import Path
from pprint import pprint

import glom
import ipywidgets as widgets
import requests
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

CHAT_API_URL = "http://localhost:8000/chat"
AUTH_API_URL = "http://localhost:8000/auth/sign-in"
TERMINAL_LINE_WIDTH = 80
MAX_HISTORY_LENGTH = 10
USERNAME = os.getenv("SUPPORT_USERNAME")
PASSWORD = os.getenv("SUPPORT_PASSWORD")

In [ ]:
def authenticate(username, password):
    response = requests.post(AUTH_API_URL, json={"login": username, "password": password})
    response.raise_for_status()
    return glom.glom(response.json(), "access_token")


API_TOKEN = authenticate(USERNAME, PASSWORD)
API_TOKEN

In [ ]:
def printf(message):
    print(textwrap.fill(message, width=TERMINAL_LINE_WIDTH))


class ChatService:
    def __init__(self):
        self.history = []
        self.counter = 1

    def reset(self):
        self.history = []
        self.counter = 1

    def chat(self, message, include_attachment=False):
        request = {
            "message": message,
            "history": [{"role": role, "content": content} for role, content in self.history[-MAX_HISTORY_LENGTH:]],
        }

        if include_attachment:
            coin_image_path = Path.cwd() / "../tests/images/coin1.jpg"
            coin_image_path = coin_image_path.resolve()
            with open(coin_image_path, "rb") as f:
                image_data = f.read()
            request["attachment"] = {
                "filename": "coin.jpg",
                "content_type": "image/jpeg",
                "data": base64.b64encode(image_data).decode("utf-8"),
            }

        print("\nRequest:")
        pprint(request)
        print()

        response = requests.post(CHAT_API_URL, json=request, headers={"Authorization": f"Bearer {API_TOKEN}"})
        response.raise_for_status()

        data = response.json()
        answer = data.get("message", "")
        self.history.append(("user", message))
        self.history.append(("assistant", answer))
        self.counter += 1
        answer = textwrap.fill(answer, width=TERMINAL_LINE_WIDTH)
        return answer

In [ ]:
chat = ChatService()

output = widgets.Output()

text_field = widgets.Textarea(value="", placeholder="Ask anything...", disabled=False)
include_attachment_checkbox = widgets.Checkbox(value=False, description="Include coin image", disabled=False)

button = widgets.Button(description="Send", button_style="", tooltip="Click to run logic", icon="comment")

clear_button = widgets.Button(
    description="Clear",
    button_style="warning",
    tooltip="Start new session",
    icon="archive",
)


def on_button_click(b):
    with output:
        question = text_field.value.strip()
        print(f">>> You: {question}")
        if not question:
            print(">>> Please enter a question.")
            return

        answer = chat.chat(question, include_attachment_checkbox.value)
        text_field.value = ""
        include_attachment_checkbox.value = False
        print(f"<<< Assistant: {answer}\n\n")


def on_clear_click(b):
    text_field.value = ""
    output.clear_output()
    chat.reset()


button.on_click(on_button_click)
clear_button.on_click(on_clear_click)

button_box = widgets.HBox([button, clear_button])
display(text_field, include_attachment_checkbox, button_box, output)